# SALD on a 2D OU-Gaussian Mixture

This notebook validates the Euler-Maruyama implementation of SALD for $r\in\{1,2,4,10,50,100\}$.

The empirical trend of interest is that the terminal mismatch

$$
KL(\rho_{rT} \| \pi_T)
$$

should shrink as $r$ grows. The theorem we want to compare against gives the upper bound

$$
KL(\rho_{rT} \| \pi_T)
\le
\exp\left(- r \int_0^T C^{\mathcal{LSI}}_t \, dt\right)
\exp\left(\frac{T}{2 r \alpha}\right)
KL(\rho_0 \| \pi_0)
+
\frac{T}{2r}
\exp\left(\frac{T}{2 r \alpha}\right)
\mathcal{A}_{\alpha},
$$

so the practical questions are:

- Does the final KL decrease as $r$ increases?
- In the large-$r$ regime, does the curve behave roughly like a $1/r$ shrinkage law before hitting a discretization / finite-sample floor?

The experimental setup in this notebook is:

- The target $\pi_T = p_{\mathrm{data}}$ is a 2D Gaussian mixture with two clearly separated means on the $x_1$ axis and unit covariance per component.
- The forward process is the OU / VP diffusion with a linear noise schedule $\beta(\tau)$.
- The reverse initial law is a centered Gaussian.
- The main quantitative metric is a discrete KL on the informative $x_1$ marginal, because all nontrivial multimodality lives on that axis.
- The 2D sample figure augments the simulated $x_1$ samples with an independent $x_2 \sim \mathcal{N}(0,1)$, which is exact here because the second coordinate stays standard Gaussian.


## Run This Cell First

This cell pins common CPU thread pools to `1` and optionally binds the run to a chosen GPU.  To select a GPU, set `SALD_PHYSICAL_GPU` before starting the notebook, or use the command-line runner `python3 run_reproduce.py --gpu ID`.  To force CPU execution, set `SALD_FORCE_CPU=1`.

Important: this must be the first executed cell in the kernel. If `torch` has already been imported, restart the kernel first.


In [ ]:
import os

# Device selection for reproducibility.  By default we do not force a physical
# GPU id; CUDA users may set SALD_PHYSICAL_GPU or run `python3 run_reproduce.py --gpu ID`.
if os.environ.get("SALD_FORCE_CPU", "0") == "1":
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
elif "SALD_PHYSICAL_GPU" in os.environ:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ["SALD_PHYSICAL_GPU"]

for key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ.setdefault(key, "1")

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("CPU threads are pinned to 1 for OMP/MKL/OpenBLAS/NumExpr")


## VP Diffusion and Closed-Form Structure

The forward process used throughout this notebook is the VP diffusion

$$
\mathrm{d} Y_\tau = -\frac12\beta(\tau)Y_\tau\mathrm{d}\tau + \sqrt{\beta(\tau)}\mathrm{d} W_\tau,
\qquad
\beta(\tau)=\beta_{\min}+\frac{\beta_{\max}-\beta_{\min}}{T}\tau.
$$

For this VP process, the reverse-indexed coefficients are

$$
B_t(x) = -\frac12\beta(T-t)x,
\qquad
\sigma_t^2=\beta(T-t),
\qquad
u_t(x)=\frac12\beta(T-t)\left(x+\nabla\log p_t(x)\right).
$$

In the first 2D example, the forward marginals preserve a simple form: a bimodal Gaussian mixture along $x_1$ and a standard Gaussian along $x_2$. Let

$$
m(t) = \alpha(T-t), \qquad
\alpha(\tau) = \exp\left(
    -\frac12 \int_0^\tau \beta(u) \, du
\right),
$$

and let the terminal data distribution have means at $x_1 = \pm \mu$. Then the reverse-indexed marginals can be written as

$$
p_t(x_1, x_2)
=
\left[
    \frac12 \varphi(x_1 - \mu m(t))
    +
    \frac12 \varphi(x_1 + \mu m(t))
\right]
\varphi(x_2),
$$

where $\varphi$ is the one-dimensional standard Gaussian density. The only nontrivial score term is therefore on $x_1$, and it has the closed form

$$
\partial_{x_1} \log p_t(x_1)
=
-x_1 + \mu m(t) \tanh(\mu m(t) x_1).
$$

The unguided SALD baseline implements the Euler-Maruyama update

$$
X_{k+1}
=
X_k + \eta \, \nabla \log p_{t(k\eta)}(X_k)
+ \sqrt{2\eta}\,\xi_k,
\qquad
t(s)=s/r.
$$

To keep the reverse starting law close to a centered Gaussian, the default schedule uses $T=1$, $\beta_{\min}=0.1$, and $\beta_{\max}=14$, which gives

$$
\alpha(T) \approx 0.029,
$$

so $p_0$ is already very close to a centered Gaussian while the terminal target $p_T=p_{\mathrm{data}}$ still has two visually distinct modes.

In [ ]:
from pathlib import Path
from pprint import pprint

import pandas as pd

from sald_em_validation import (
    ModelConfig,
    RunProfile,
    alpha_tau_scalar,
    configure_torch,
    make_overview_figure,
    make_sample_grid,
    plot_target_family,
    run_experiment_suite,
    runtime_report,
    set_plot_style,
)

device = configure_torch(seed=0)
set_plot_style()
pprint(runtime_report())

OUTPUT_DIR = Path(os.environ.get("SALD_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Centralized experiment configuration:
# adjust r, mean separation, particle count, plot ranges, and other knobs here.
EXPERIMENT = {
    "run_name": "safe_mu225",
    "seed": 123,
    "r_values": [1, 2, 4, 10, 50, 100],
    "selected_r_for_plots": [1, 2, 4, 10, 50, 100],
    "T": 1.0,
    "beta_min": 0.1,
    "beta_max": 14.0,
    "target_x_mean_abs": 2.25,
    "n_particles": 10_000,
    "eta_s": 0.001,
    "hist_bins": 768,
    "checkpoint_count": 33,
    "scatter_points": 4_000,
    "target_family_xlim": (-6.0, 6.0),
    "final_density_xlim": (-6.0, 6.0),
    "sample_xlim": (-5.8, 5.8),
    "sample_ylim": (-3.25, 3.25),
}

model_cfg = ModelConfig(
    T=EXPERIMENT["T"],
    beta_min=EXPERIMENT["beta_min"],
    beta_max=EXPERIMENT["beta_max"],
    x_mean=EXPERIMENT["target_x_mean_abs"],
)

profile = RunProfile(
    name=EXPERIMENT["run_name"],
    n_particles=EXPERIMENT["n_particles"],
    eta_s=EXPERIMENT["eta_s"],
    hist_bins=EXPERIMENT["hist_bins"],
    checkpoint_count=EXPERIMENT["checkpoint_count"],
    scatter_points=EXPERIMENT["scatter_points"],
    x_min=EXPERIMENT["final_density_xlim"][0],
    x_max=EXPERIMENT["final_density_xlim"][1],
)

r_values = EXPERIMENT["r_values"]
scatter_r = EXPERIMENT["selected_r_for_plots"]
RUN_NAME = EXPERIMENT["run_name"]

alpha_T = alpha_tau_scalar(model_cfg.T, model_cfg)

pd.DataFrame(
    [{"parameter": k, "value": v} for k, v in EXPERIMENT.items()]
    + [{"parameter": "alpha(T)", "value": alpha_T}]
)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.0, 4.6), constrained_layout=True)
plot_target_family(ax, model_cfg, x_lim=EXPERIMENT["target_family_xlim"])
fig.savefig(OUTPUT_DIR / "target_family.png", dpi=180, bbox_inches="tight")
plt.show()


## Experimental Design

The comparison is intentionally fair and conservative:

- Every $r$ uses the same initial sample batch $X_0 \sim \mathcal{N}(0,1)$.
- Every $r$ uses the same SALD physical stepsize $\eta_s$, so this is a comparison inside one discretized continuous-time family rather than a deliberately coarsened integrator for large $r$.
- There is no dataloader, no worker pool, and no multiprocessing; the full run uses a single Python process.
- The main metric is the discrete KL on the $x_1$ marginal, which is exactly where the bimodality lives.
- All tunable experiment hyperparameters are collected in the single configuration cell above.

If you want a stronger visual separation of the two modes, the main knob is `target_x_mean_abs`. If you want a lower large-$r$ error floor, the most direct knobs are `n_particles`, `eta_s`, and `hist_bins`.


In [ ]:
suite = run_experiment_suite(
    r_values=r_values,
    cfg=model_cfg,
    profile=profile,
    seed=EXPERIMENT["seed"],
    verbose=True,
)

summary_df = suite["summary_df"].copy()
summary_df["r_times_kl"] = summary_df["r"] * summary_df["kl_to_target"]

summary_path = OUTPUT_DIR / f"sald_summary_{RUN_NAME}.csv"
traj_path = OUTPUT_DIR / f"sald_trajectory_{RUN_NAME}.csv"
summary_df.to_csv(summary_path, index=False)
suite["trajectory_df"].to_csv(traj_path, index=False)

summary_df[
    ["r", "kl_to_target", "r_times_kl", "var", "n_steps", "wall_clock_sec"]
]


In [ ]:
fig = make_overview_figure(
    suite,
    selected_r=scatter_r,
    target_family_xlim=EXPERIMENT["target_family_xlim"],
    final_density_xlim=EXPERIMENT["final_density_xlim"],
)
fig.savefig(OUTPUT_DIR / f"sald_overview_{RUN_NAME}.png", dpi=220, bbox_inches="tight")
fig


In [ ]:
fig = make_sample_grid(
    suite,
    selected_r=scatter_r,
    seed=EXPERIMENT["seed"],
    x_lim=EXPERIMENT["sample_xlim"],
    y_lim=EXPERIMENT["sample_ylim"],
)
fig.savefig(OUTPUT_DIR / f"sald_samples_{RUN_NAME}.png", dpi=220, bbox_inches="tight")
fig


## How To Read The Figures

The usual qualitative pattern is:

- The terminal-target KL $KL(\rho_s \| \pi_T)$ stays visibly higher along the trajectory for $r=1$, and the terminal sample cloud struggles to develop the two target modes.
- As $r$ increases, the terminal samples move closer to the two-mode target $\pi_T = p_{\mathrm{data}}$.
- The final KL curve versus $r$ should decrease overall; in the large-$r$ regime it often reaches a floor driven by Euler discretization and finite particle count.

If you want to push that floor lower, the most direct changes are:

- Increase `n_particles`.
- Decrease `eta_s`.
- Increase `hist_bins`.
- Optionally add more $r$ values inside `r_values`.

All outputs are written into the local `outputs/` directory: the summary CSV, the trajectory CSV, and the main figures.


## Guided SALD on Tilted Families

We now extend the experiment from the prior family $\pi_t$ to the guided tilted family

$$
q_t^X(x) \propto \pi_t(x) \exp(-f(x)).
$$

The SALD dynamics become

$$
dX_s = \nabla \log q_{t(s)}^X(X_s) \, ds + \sqrt{2} \, dW_s,
\qquad t(s) = s / r.
$$

so the score is the prior score plus the extra guidance term $-\nabla f(x)$.

In this section the guidance is time-invariant and is built from a two-moons reference distribution. We keep the prior family $\pi_t$ exactly as before and only change the moving target to $q_t^X$.


In [ ]:
from sald_guided_validation import (
    GuidedFieldConfig,
    GuidedRunProfile,
    make_guided_lambda_heatmap,
    make_guided_overview_figure,
    make_guided_sample_grid,
    prepare_guidance_field,
    run_guided_experiment_suite,
    run_guided_lambda_sweep,
)


### Guidance Design

We define the guidance potential as

$$
f(x) = \frac{1}{\lambda} \cdot \frac{1}{N} \sum_{j=1}^N \|x - y_j\|_2,
$$

where the $y_j$ are samples from a translated and rescaled two-moons distribution. Its gradient is

$$
\nabla f(x) = \frac{1}{\lambda} \cdot \frac{1}{N} \sum_{j=1}^N \frac{x - y_j}{\|x - y_j\|_2}.
$$

A literal $N = 5{,}000{,}000$ average inside every SALD step is computationally intractable. The notebook therefore uses the correct numerical surrogate for the same population objective in 2D:

- build a fixed Monte Carlo reference cloud for the two moons,
- precompute $f$ and $\nabla f$ on a 2D grid once,
- use bilinear interpolation during SALD sampling.

The KL is now computed against the guided terminal target $q_T^X$, not against $\pi_T$.


In [ ]:
GUIDED_EXPERIMENT = {
    'run_name': 'guided_moons_lam1',
    'seed': 321,
    "r_values": [1, 2, 4, 10, 50, 100],
    "selected_r_for_plots": [1, 2, 4, 10, 50, 100],
    'selected_r_for_overview': [1, 10, 50, 100],
    'lambda_guidance': 1.0,
    'T': EXPERIMENT['T'],
    'beta_min': EXPERIMENT['beta_min'],
    'beta_max': EXPERIMENT['beta_max'],
    'target_x_mean_abs': EXPERIMENT['target_x_mean_abs'],
    "n_particles": 10_000,
    'eta_s': 0.001,
    'checkpoint_count': 25,
    'scatter_points': 6_000,
    'n_moon_requested': 5_000_000,
    'n_moon_reference': 16_384,
    'moon_noise': 0.18,
    'moon_scale': 2.05,
    'moon_shift_x': -1.05,
    'moon_shift_y': -0.25,
    'field_bins_x': 160,
    'field_bins_y': 128,
    'field_x_min': -5.4,
    'field_x_max': 5.4,
    'field_y_min': -3.3,
    'field_y_max': 3.3,
    'distance_eps': 1e-3,
    'grid_chunk_size': 1024,
    'ref_chunk_size': 2048,
    'plot_reference_points': 12_000,
}

guided_model_cfg = ModelConfig(
    T=GUIDED_EXPERIMENT['T'],
    beta_min=GUIDED_EXPERIMENT['beta_min'],
    beta_max=GUIDED_EXPERIMENT['beta_max'],
    x_mean=GUIDED_EXPERIMENT['target_x_mean_abs'],
)

guided_field_cfg = GuidedFieldConfig(
    lambda_guidance=GUIDED_EXPERIMENT['lambda_guidance'],
    n_reference=GUIDED_EXPERIMENT['n_moon_reference'],
    moon_noise=GUIDED_EXPERIMENT['moon_noise'],
    moon_scale=GUIDED_EXPERIMENT['moon_scale'],
    moon_shift_x=GUIDED_EXPERIMENT['moon_shift_x'],
    moon_shift_y=GUIDED_EXPERIMENT['moon_shift_y'],
    x_min=GUIDED_EXPERIMENT['field_x_min'],
    x_max=GUIDED_EXPERIMENT['field_x_max'],
    y_min=GUIDED_EXPERIMENT['field_y_min'],
    y_max=GUIDED_EXPERIMENT['field_y_max'],
    bins_x=GUIDED_EXPERIMENT['field_bins_x'],
    bins_y=GUIDED_EXPERIMENT['field_bins_y'],
    distance_eps=GUIDED_EXPERIMENT['distance_eps'],
    ref_chunk_size=GUIDED_EXPERIMENT['ref_chunk_size'],
    grid_chunk_size=GUIDED_EXPERIMENT['grid_chunk_size'],
    plot_reference_points=GUIDED_EXPERIMENT['plot_reference_points'],
)

guided_profile = GuidedRunProfile(
    name=GUIDED_EXPERIMENT['run_name'],
    n_particles=GUIDED_EXPERIMENT['n_particles'],
    eta_s=GUIDED_EXPERIMENT['eta_s'],
    checkpoint_count=GUIDED_EXPERIMENT['checkpoint_count'],
    scatter_points=GUIDED_EXPERIMENT['scatter_points'],
    x_min=GUIDED_EXPERIMENT['field_x_min'],
    x_max=GUIDED_EXPERIMENT['field_x_max'],
    y_min=GUIDED_EXPERIMENT['field_y_min'],
    y_max=GUIDED_EXPERIMENT['field_y_max'],
    bins_x=GUIDED_EXPERIMENT['field_bins_x'],
    bins_y=GUIDED_EXPERIMENT['field_bins_y'],
)

GUIDED_RUN_NAME = GUIDED_EXPERIMENT['run_name']
guided_r_values = GUIDED_EXPERIMENT['r_values']
guided_plot_r = GUIDED_EXPERIMENT['selected_r_for_plots']
guided_overview_r = GUIDED_EXPERIMENT['selected_r_for_overview']

pd.DataFrame([
    {'parameter': k, 'value': v} for k, v in GUIDED_EXPERIMENT.items()
])


In [ ]:
guided_state = prepare_guidance_field(
    guided_field_cfg,
    seed=GUIDED_EXPERIMENT['seed'],
    verbose=True,
)

guided_suite = run_guided_experiment_suite(
    guided_r_values,
    guided_model_cfg,
    guided_profile,
    guided_state,
    seed=GUIDED_EXPERIMENT['seed'],
    verbose=True,
)

guided_summary_df = guided_suite['summary_df'].copy()
guided_summary_df['r_times_kl'] = guided_summary_df['r'] * guided_summary_df['kl_to_target']

guided_summary_path = OUTPUT_DIR / f'guided_summary_{GUIDED_RUN_NAME}.csv'
guided_traj_path = OUTPUT_DIR / f'guided_trajectory_{GUIDED_RUN_NAME}.csv'
guided_summary_df.to_csv(guided_summary_path, index=False)
guided_suite['trajectory_df'].to_csv(guided_traj_path, index=False)

guided_summary_df[
    ['r', 'kl_to_target', 'r_times_kl', 'mean_guidance', 'mean_penalty', 'n_steps', 'wall_clock_sec']
]


In [ ]:
guided_overview_fig = make_guided_overview_figure(
    guided_suite,
    selected_r=guided_overview_r,
)
guided_overview_fig.savefig(
    OUTPUT_DIR / f'guided_overview_{GUIDED_RUN_NAME}.png',
    dpi=220,
    bbox_inches='tight',
)
guided_overview_fig


In [ ]:
guided_samples_fig = make_guided_sample_grid(
    guided_suite,
    selected_r=guided_plot_r,
    x_lim=(GUIDED_EXPERIMENT['field_x_min'], GUIDED_EXPERIMENT['field_x_max']),
    y_lim=(GUIDED_EXPERIMENT['field_y_min'], GUIDED_EXPERIMENT['field_y_max']),
)
guided_samples_fig.savefig(
    OUTPUT_DIR / f'guided_samples_{GUIDED_RUN_NAME}.png',
    dpi=220,
    bbox_inches='tight',
)
guided_samples_fig


## Velocity-Aware SALD

This section uses the same VP diffusion and reverse-indexed marginal family as the SALD implementation above. Write the reverse marginals as $p_t = q_{T-t}$, with score $\nabla\log p_t$ implemented by `prior_score_2d` and the same `model_cfg` linear VP schedule:

$$
\mathrm{d} Y_\tau = -\frac12\beta(\tau)Y_\tau\mathrm{d}\tau + \sqrt{\beta(\tau)}\mathrm{d} W_\tau,
\qquad
\beta(\tau)=\beta_{\min}+\frac{\beta_{\max}-\beta_{\min}}{T}\tau.
$$

For this VP process,

$$
B_t(x) = -\frac12\beta(T-t)x,
\qquad
\sigma_t^2=\beta(T-t),
\qquad
u_t(x)=\frac12\beta(T-t)\left(x+\nabla\log p_t(x)\right).
$$

The guided moving target is

$$
\pi_t(x) \propto p_t(x)\exp(-f(x)),
$$

where the guidance potential $f$ is the same time-invariant two-moons potential used in the previous guided section. With $t(s)=s/r$ and $\dot t(s)=1/r$, Velocity-Aware SALD is

$$
\mathrm{d} X_s =
\left(
\dot t(s)u_{t(s)}(X_s)
+\frac{\sigma_{t(s)}^2}{2}\nabla\log p_{t(s)}(X_s)
-\frac{\sigma_{t(s)}^2}{2}\nabla f(X_s)
\right)\mathrm{d} s
+\sigma_{t(s)}\mathrm{d} W_s.
$$

Substituting the VP coefficients gives the concrete dynamics implemented below:

$$
\mathrm{d} X_s =
\frac{\beta(T-t(s))}{2}
\left(
\frac{X_s}{r}
+\left(1+\frac1r\right)\nabla\log p_{t(s)}(X_s)
-\nabla f(X_s)
\right)\mathrm{d} s
+\sqrt{\beta(T-t(s))}\mathrm{d} W_s.
$$

Equivalently, one Euler-Maruyama step with physical stepsize $\eta$ is

$$
X_{k+1}=X_k+
\eta\frac{\beta(T-t_k)}{2}
\left(
\frac{X_k}{r}
+\left(1+\frac1r\right)\nabla\log p_{t_k}(X_k)
-\nabla f(X_k)
\right)
+\sqrt{\eta\beta(T-t_k)}\xi_k,
\qquad
t_k=\frac{k\eta}{r}.
$$

The $X_s/r$ and $(1+1/r)\nabla\log p_t$ terms come from the VP velocity field plus the Langevin correction. The target itself is not the $r$-dependent auxiliary tilt; it is the VA-SALD target $\pi_t \propto p_t\exp(-f)$, so the terminal KL is directly comparable with the previous guided section.


In [ ]:
from sald_velocity_aware_validation import (
    make_velocity_aware_guided_lambda_heatmap,
    make_velocity_aware_guided_overview_figure,
    make_velocity_aware_guided_sample_grid,
    run_velocity_aware_guided_experiment_suite,
    run_velocity_aware_guided_lambda_sweep,
)


In [ ]:
from dataclasses import replace

VA_EXPERIMENT = dict(GUIDED_EXPERIMENT)
VA_EXPERIMENT.update({
    'run_name': 'velocity_aware_moons_lam1',
})

# Fair-comparison handles: VA-SALD uses the same reverse-indexed VP family,
# the same Monte Carlo guidance surrogate, the same initial sample seed, and
# the same r/stepsize grid as the guided SALD section immediately above.
va_model_cfg = guided_model_cfg
va_field_cfg = guided_field_cfg
va_profile = replace(guided_profile, name=VA_EXPERIMENT['run_name'])

VA_RUN_NAME = VA_EXPERIMENT['run_name']
va_r_values = VA_EXPERIMENT['r_values']
va_plot_r = VA_EXPERIMENT['selected_r_for_plots']
va_overview_r = VA_EXPERIMENT['selected_r_for_overview']

pd.DataFrame([
    {'parameter': k, 'value': v} for k, v in VA_EXPERIMENT.items()
])


In [ ]:
if 'guided_state' in globals() and guided_field_cfg == va_field_cfg:
    va_state = guided_state
    print('Reusing guided_state so VA-SALD uses the same guidance function as guided SALD.')
else:
    va_state = prepare_guidance_field(
        va_field_cfg,
        seed=VA_EXPERIMENT['seed'],
        verbose=True,
    )

va_suite = run_velocity_aware_guided_experiment_suite(
    va_r_values,
    va_model_cfg,
    va_profile,
    va_state,
    seed=VA_EXPERIMENT['seed'],
    verbose=True,
)

va_summary_df = va_suite['summary_df'].copy()
va_summary_df['r_times_kl'] = va_summary_df['r'] * va_summary_df['kl_to_target']

va_summary_path = OUTPUT_DIR / f'velocity_aware_summary_{VA_RUN_NAME}.csv'
va_traj_path = OUTPUT_DIR / f'velocity_aware_trajectory_{VA_RUN_NAME}.csv'
va_summary_df.to_csv(va_summary_path, index=False)
va_suite['trajectory_df'].to_csv(va_traj_path, index=False)

va_summary_df[
    ['r', 'kl_to_target', 'r_times_kl', 'mean_guidance', 'mean_penalty', 'mean_beta_reverse', 'mean_sigma_reverse', 'n_steps', 'wall_clock_sec']
]


In [ ]:
va_overview_fig = make_velocity_aware_guided_overview_figure(
    va_suite,
    selected_r=va_overview_r,
)
va_overview_fig.savefig(
    OUTPUT_DIR / f'velocity_aware_overview_{VA_RUN_NAME}.png',
    dpi=220,
    bbox_inches='tight',
)
va_overview_fig


In [ ]:
va_samples_fig = make_velocity_aware_guided_sample_grid(
    va_suite,
    selected_r=va_plot_r,
    x_lim=(VA_EXPERIMENT['field_x_min'], VA_EXPERIMENT['field_x_max']),
    y_lim=(VA_EXPERIMENT['field_y_min'], VA_EXPERIMENT['field_y_max']),
)
va_samples_fig.savefig(
    OUTPUT_DIR / f'velocity_aware_samples_{VA_RUN_NAME}.png',
    dpi=220,
    bbox_inches='tight',
)
va_samples_fig


In [ ]:
# Comparison: previous guided SALD vs Velocity-Aware SALD on the same VP family
if 'guided_summary_df' in dir() and 'va_summary_df' in dir():
    compare_df = guided_summary_df[['r', 'kl_to_target']].rename(columns={'kl_to_target': 'guided_kl'})
    compare_df = compare_df.merge(
        va_summary_df[['r', 'kl_to_target']].rename(columns={'kl_to_target': 'velocity_aware_kl'}),
        on='r',
        how='inner',
    )
    compare_df['kl_ratio'] = compare_df['velocity_aware_kl'] / compare_df['guided_kl']
    display(compare_df[['r', 'guided_kl', 'velocity_aware_kl', 'kl_ratio']])
else:
    print('Run both the guided and Velocity-Aware SALD sections first.')


### What To Look For in the Velocity-Aware Experiment

This section changes only the reverse-time dynamics. It keeps the VP diffusion family, guidance field, initial sample seed, $r$ grid, stepsize, and terminal target aligned with the previous guided section.

Key structural checks:

- **Same VP family**: `va_model_cfg = guided_model_cfg`, and the score is still `prior_score_2d(x, t, va_model_cfg)`, so $\nabla\log p_t$ is the same reverse-indexed VP score used above.
- **VP-aware drift scale**: the deterministic update is multiplied by $\beta(T-t)/2$, not by the OU decay factor $\alpha(t)$.
- **VP-aware noise scale**: the stochastic update uses $\sqrt{\eta\beta(T-t)}\xi_k$, matching $\sigma_t^2=\beta(T-t)$.
- **Velocity terms are not a target tilt**: the $X_s/r$ and $(1+1/r)\nabla\log p_t$ terms come from $\dot t(s)u_t$ plus the Langevin score correction. The target remains $\pi_t\propto p_t\exp(-f)$.
- **Same guide and initialization**: `va_field_cfg = guided_field_cfg`, `va_state = guided_state` during full-notebook execution, and `VA_EXPERIMENT` copies `GUIDED_EXPERIMENT`, so both sections use the same $f$, initial sample seed, $r$ grid, and stepsize.
- **Direct KL comparison**: the final KL is against the same guided terminal target $\pi_T\propto p_T\exp(-f)$ as the previous guided SALD section.

Diagnostics:

- Compare `guided_kl` and `velocity_aware_kl` in the table above at matching $r$ values.
- Inspect `mean_beta_reverse`: it follows the same reverse VP schedule $\beta(T-t)$, starting near $\beta_{\max}$ and ending near $\beta_{\min}$.
- The guidance scale is fixed at $\lambda=1$ here; edit `lambda_guidance` in `GUIDED_EXPERIMENT` only when intentionally changing the target.


## DOIT Adaptation on the Same Guided VP Family

The DOIT baseline is implemented self-contained in `sald_doit_validation.py`, following the training-free Doob $h$-transform estimator used by DOIT. No external repository clone is required for this supplementary code.

For a base reverse sampler with transition density $\phi_\theta(x_{\ell-1}\mid x_\ell)$ and an event or reward-induced preference encoded by $h(x_0,0)$, DOIT defines

$h(x_\ell,t_\ell)=\mathbb{E}[h(X_0,0)\mid X_{t_\ell}=x_\ell].$

The Doob-transformed process adds the dynamic correction $\nabla \log h$ to the base reverse dynamics. In the discrete Gaussian transition used by the implementation, the key plug-in estimator is

$\widehat{\nabla \log h}(x_\ell,t_\ell)=\frac{\sum_{m=1}^{M} h(x_0^{(m)},0)\nabla_{x_\ell}\log \phi_\theta(x_{\ell-1}^{(m)}\mid x_\ell)}{\max\left(\sum_{m=1}^{M} h(x_0^{(m)},0),\eta_{t_\ell}\right)}.$

For the comparison here, DOIT is not slowed down. The budget label $r_b\in\{1,2,4,10,50,100\}$ only determines how many Euler transitions are allowed:

$N_{\mathrm{DOIT}}(r_b)=\left\lceil \frac{r_bT}{\eta_s}\right\rceil,\qquad \eta_{\mathrm{DOIT}}=\frac{T}{N_{\mathrm{DOIT}}(r_b)},\qquad t_k=k\eta_{\mathrm{DOIT}}.$

Thus DOIT traverses the original VP reverse interval $t\in[0,T]$ for every budget. It does not use $t=k\eta/r_b$ and it does not put $r_b$ into the base drift. The unconditioned base sampler must use the original VP reverse SDE drift, not the continuity velocity $u_t$. For the same two-Gaussian marginal family this drift is:

$b_{\mathrm{base}}(x,t_k)=\frac{\beta(T-t_k)}{2}x+\beta(T-t_k)\nabla\log p_{t_k}(x)=\frac{\beta(T-t_k)}{2}\left(x+2\nabla\log p_{t_k}(x)\right).$

The released code uses a softmax-weighted local-search estimator. In this VP adaptation we use Boltzmann weights without per-particle reward-standard-deviation normalization, because that normalization changes the effective guidance strength across methods. For each particle, DOIT draws $M$ local proposals

$x_{k+1}^{(m)}=x_k+\eta_{\mathrm{DOIT}}\, b_{\mathrm{base}}(x_k,t_k)+\sqrt{\eta_{\mathrm{DOIT}}\beta(T-t_k)}\,z_m,\qquad z_m\sim\mathcal{N}(0,I).$

It scores them by the reward $R(x)=-f(x)$. Because the guided target is proportional to $\exp(-f)$, lower penalty $f$ is better; equivalently the reported guidance objective $R=-f$ should increase for all three algorithms. The weights are

$w_m=\mathrm{softmax}_m\left(\frac{R(x_{k+1}^{(m)})-\max_j R(x_{k+1}^{(j)})}{\tau_R}\right),\qquad \widehat{g}_{\mathrm{Doob}}=\sum_{m=1}^{M} w_m \frac{z_m}{\sqrt{\eta_{\mathrm{DOIT}}\beta(T-t_k)}}.$

The adapted DOIT update is then

$X_{k+1}=X_k+\eta_{\mathrm{DOIT}}\left(b_{\mathrm{base}}(X_k,t_k)+\gamma\beta(T-t_k)\widehat{g}_{\mathrm{Doob}}\right)+\sqrt{\eta_{\mathrm{DOIT}}\beta(T-t_k)}\,\xi_k.$

For fair computational budget at each label $r_b$, SALD and VA-SALD use $N(r_b)=\lceil r_bT/\eta_s\rceil$ slowed physical steps with $t=s/r_b$, while DOIT uses the same number of VP reverse transitions over $[0,T]$. Because DOIT spends $M$ proposal evaluations per particle and step, the DOIT profile uses approximately `n_particles / M` particles, so `effective_budget_particles = n_particles * M` matches the SALD and VA-SALD particle-step budget.

In [ ]:
from sald_doit_validation import (
    DoitConfig,
    equal_budget_doit_profile,
    make_doit_overview_figure,
    make_three_algorithm_comparison_figure,
    run_doit_guided_experiment_suite,
)


In [ ]:
DOIT_EXPERIMENT = dict(GUIDED_EXPERIMENT)
DOIT_EXPERIMENT.update({
    'run_name': 'doit_moons_lam1',
    'doit_m_proposals': 4,
    'doit_tau': 0.6,
    'doit_gamma': 1.0,
})

doit_cfg = DoitConfig(
    m_proposals=DOIT_EXPERIMENT['doit_m_proposals'],
    tau=DOIT_EXPERIMENT['doit_tau'],
    gamma=DOIT_EXPERIMENT['doit_gamma'],
)
doit_profile = equal_budget_doit_profile(
    guided_profile,
    doit_cfg,
    name=DOIT_EXPERIMENT['run_name'],
)

DOIT_RUN_NAME = DOIT_EXPERIMENT['run_name']
doit_r_values = DOIT_EXPERIMENT['r_values']

pd.DataFrame([
    {'parameter': k, 'value': v} for k, v in DOIT_EXPERIMENT.items()
] + [
    {'parameter': 'doit_particles_equal_budget', 'value': doit_profile.n_particles},
    {'parameter': 'sald_va_particles', 'value': guided_profile.n_particles},
])


In [ ]:
doit_suite = run_doit_guided_experiment_suite(
    doit_r_values,
    guided_model_cfg,
    doit_profile,
    guided_state,
    doit_cfg,
    seed=DOIT_EXPERIMENT['seed'],
    verbose=True,
)

doit_summary_df = doit_suite['summary_df'].copy()
doit_summary_df['r_times_kl'] = doit_summary_df['r'] * doit_summary_df['kl_to_target']

doit_summary_path = OUTPUT_DIR / f'doit_summary_{DOIT_RUN_NAME}.csv'
doit_traj_path = OUTPUT_DIR / f'doit_trajectory_{DOIT_RUN_NAME}.csv'
doit_summary_df.to_csv(doit_summary_path, index=False)
doit_suite['trajectory_df'].to_csv(doit_traj_path, index=False)

doit_summary_df[
    ['r', 'kl_to_target', 'r_times_kl', 'mean_guidance', 'mean_penalty', 'n_particles', 'effective_budget_particles', 'n_steps', 'wall_clock_sec']
]


In [ ]:
doit_overview_fig = make_doit_overview_figure(
    doit_suite,
    selected_r=guided_overview_r,
)
doit_overview_fig.savefig(
    OUTPUT_DIR / f'doit_overview_{DOIT_RUN_NAME}.png',
    dpi=240,
    bbox_inches='tight',
)
doit_overview_fig

In [ ]:
two_moon_compare_fig = make_three_algorithm_comparison_figure(
    guided_suite,
    va_suite,
    doit_suite,
    title_prefix='Two-Moons Guided 2-Gaussian VP',
    selected_r=guided_r_values,
)
two_moon_compare_fig.savefig(
    OUTPUT_DIR / f'three_algorithm_comparison_two_moons_{GUIDED_RUN_NAME}.png',
    dpi=240,
    bbox_inches='tight',
)
two_moon_compare_fig


In [ ]:
if 'guided_summary_df' in dir() and 'va_summary_df' in dir() and 'doit_summary_df' in dir():
    three_way_df = guided_summary_df[['r', 'kl_to_target', 'mean_guidance', 'mean_penalty']].rename(
        columns={'kl_to_target': 'sald_kl', 'mean_guidance': 'sald_mean_objective', 'mean_penalty': 'sald_mean_penalty'}
    )
    three_way_df = three_way_df.merge(
        va_summary_df[['r', 'kl_to_target', 'mean_guidance', 'mean_penalty']].rename(
            columns={'kl_to_target': 'va_sald_kl', 'mean_guidance': 'va_sald_mean_objective', 'mean_penalty': 'va_sald_mean_penalty'}
        ),
        on='r',
    )
    three_way_df = three_way_df.merge(
        doit_summary_df[['r', 'kl_to_target', 'mean_guidance', 'mean_penalty']].rename(
            columns={'kl_to_target': 'doit_kl', 'mean_guidance': 'doit_mean_objective', 'mean_penalty': 'doit_mean_penalty'}
        ),
        on='r',
    )
    display(three_way_df)
else:
    print('Run SALD, VA-SALD, and DOIT sections first.')


## 8-Gaussian VP Diffusion with Mode-Penalty Guidance

We now replace the terminal data distribution by an 8-mode Gaussian mixture whose centers are uniformly spaced on a circle around $(0,0)$. With radius $R$ and angular offset $\theta_0=\pi/8$, define

$$
c_j=R\left(\cos\left(\theta_0+\frac{2\pi j}{8}\right),\sin\left(\theta_0+\frac{2\pi j}{8}\right)\right),\qquad j=0,\ldots,7.
$$

The terminal distribution is

$$
p_{\mathrm{data}}(x)=\frac{1}{8}\sum_{j=0}^{7}\varphi_2(x-c_j),
$$

where $\varphi_2$ is the standard two-dimensional Gaussian density. The forward process is the same VP diffusion as before,

$$
\mathrm{d}Y_\tau=-\frac{1}{2}\beta(\tau)Y_\tau\mathrm{d}\tau+\sqrt{\beta(\tau)}\mathrm{d}W_\tau,
\qquad Y_0\sim p_{\mathrm{data}}.
$$

Because the component covariance is the stationary covariance $I$, the forward marginals remain mixtures with unit covariance and decayed means:

$$
q_\tau(x)=\frac{1}{8}\sum_{j=0}^{7}\varphi_2\left(x-\alpha(\tau)c_j\right),
\qquad
\alpha(\tau)=\exp\left(-\frac{1}{2}\int_0^\tau\beta(u)\mathrm{d}u\right).
$$

The reverse-indexed family is $p_t=q_{T-t}$, so

$$
p_t(x)=\frac{1}{8}\sum_{j=0}^{7}\varphi_2\left(x-\alpha(T-t)c_j\right).
$$

The score used by all three algorithms is exact:

$$
\nabla\log p_t(x)=
\sum_{j=0}^{7}\omega_j(x,t)\left(\alpha(T-t)c_j-x\right),
\qquad
\omega_j(x,t)=
\frac{\varphi_2(x-\alpha(T-t)c_j)}{\sum_{\ell=0}^{7}\varphi_2(x-\alpha(T-t)c_\ell)}.
$$

To penalize selected modes, choose an index set $\mathcal{P}\subset\{0,\ldots,7\}$. The default is the four modes with negative first coordinate, i.e. the left half of the circle. We use the smooth potential

$$
f_{\mathcal{P}}(x)=\lambda\sum_{j\in\mathcal{P}}
\exp\left(-\frac{\|x-c_j\|^2}{2\ell_f^2}\right),
$$

with gradient

$$
\nabla f_{\mathcal{P}}(x)=
-\frac{\lambda}{\ell_f^2}
\sum_{j\in\mathcal{P}}
\exp\left(-\frac{\|x-c_j\|^2}{2\ell_f^2}\right)(x-c_j).
$$

The guided target is

$$
\pi_t(x)\propto p_t(x)\exp(-f_{\mathcal{P}}(x)).
$$

Since $f_{\mathcal{P}}$ is a penalty, lower mean $f_{\mathcal{P}}(X_s)$ indicates stronger avoidance of the selected modes; equivalently the reported guidance objective $-f_{\mathcal{P}}$ should increase. For this same $p_t$ and $f_{\mathcal{P}}$, we run SALD, VA-SALD, and DOIT with matched budgets. The only algorithmic difference is the reverse-time update rule, not the forward process, target, score, or guidance function.

For DOIT in this experiment, the same VP reverse SDE base drift is $b_{\mathrm{base}}(x,t)=\frac{\beta(T-t)}{2}x+\beta(T-t)\nabla\log p_t(x)$; the budget label does not enter this drift.

In [ ]:
from sald_eight_gaussian_validation import (
    DEFAULT_EIGHT_GAUSSIAN_CENTERS,
    DEFAULT_LEFT_HALF_MODE_INDICES,
    EightGaussianConfig,
    equal_budget_profile as equal_budget_eight_profile,
    make_algorithm_overview_figure as make_eight_gaussian_algorithm_overview_figure,
    make_comparison_figure as make_eight_gaussian_comparison_figure,
    make_eight_gaussian_grid,
    make_eight_gaussian_setup_figure,
    run_eight_gaussian_algorithm,
)


In [ ]:
EIGHT_GAUSSIAN_EXPERIMENT = {
    'run_name': 'eight_gaussian_left_penalty_lam1',
    'seed': 654,
    'r_values': GUIDED_EXPERIMENT['r_values'],
    'selected_r_for_plots': GUIDED_EXPERIMENT['selected_r_for_plots'],
    'T': GUIDED_EXPERIMENT['T'],
    'beta_min': GUIDED_EXPERIMENT['beta_min'],
    'beta_max': GUIDED_EXPERIMENT['beta_max'],
    'penalty_mode_indices': DEFAULT_LEFT_HALF_MODE_INDICES,
    'penalty_strength': 1.0,
    'penalty_width': 0.85,
    'n_particles': GUIDED_EXPERIMENT['n_particles'],
    'eta_s': GUIDED_EXPERIMENT['eta_s'],
    'checkpoint_count': GUIDED_EXPERIMENT['checkpoint_count'],
    'scatter_points': GUIDED_EXPERIMENT['scatter_points'],
    'field_bins_x': 176,
    'field_bins_y': 160,
    'field_x_min': -5.8,
    'field_x_max': 5.8,
    'field_y_min': -4.6,
    'field_y_max': 4.6,
    'doit_m_proposals': DOIT_EXPERIMENT['doit_m_proposals'],
    'doit_tau': DOIT_EXPERIMENT['doit_tau'],
    'doit_gamma': DOIT_EXPERIMENT['doit_gamma'],
}

eight_cfg = EightGaussianConfig(
    T=EIGHT_GAUSSIAN_EXPERIMENT['T'],
    beta_min=EIGHT_GAUSSIAN_EXPERIMENT['beta_min'],
    beta_max=EIGHT_GAUSSIAN_EXPERIMENT['beta_max'],
    centers=DEFAULT_EIGHT_GAUSSIAN_CENTERS,
    penalty_mode_indices=EIGHT_GAUSSIAN_EXPERIMENT['penalty_mode_indices'],
    penalty_strength=EIGHT_GAUSSIAN_EXPERIMENT['penalty_strength'],
    penalty_width=EIGHT_GAUSSIAN_EXPERIMENT['penalty_width'],
)

eight_profile = GuidedRunProfile(
    name=EIGHT_GAUSSIAN_EXPERIMENT['run_name'],
    n_particles=EIGHT_GAUSSIAN_EXPERIMENT['n_particles'],
    eta_s=EIGHT_GAUSSIAN_EXPERIMENT['eta_s'],
    checkpoint_count=EIGHT_GAUSSIAN_EXPERIMENT['checkpoint_count'],
    scatter_points=EIGHT_GAUSSIAN_EXPERIMENT['scatter_points'],
    x_min=EIGHT_GAUSSIAN_EXPERIMENT['field_x_min'],
    x_max=EIGHT_GAUSSIAN_EXPERIMENT['field_x_max'],
    y_min=EIGHT_GAUSSIAN_EXPERIMENT['field_y_min'],
    y_max=EIGHT_GAUSSIAN_EXPERIMENT['field_y_max'],
    bins_x=EIGHT_GAUSSIAN_EXPERIMENT['field_bins_x'],
    bins_y=EIGHT_GAUSSIAN_EXPERIMENT['field_bins_y'],
)

eight_doit_cfg = DoitConfig(
    m_proposals=EIGHT_GAUSSIAN_EXPERIMENT['doit_m_proposals'],
    tau=EIGHT_GAUSSIAN_EXPERIMENT['doit_tau'],
    gamma=EIGHT_GAUSSIAN_EXPERIMENT['doit_gamma'],
)
eight_doit_profile = equal_budget_eight_profile(eight_profile, eight_doit_cfg, name='eight_gaussian_doit_equal_budget')
eight_grid = make_eight_gaussian_grid(eight_profile, eight_cfg, device=device)

pd.DataFrame([
    {'parameter': k, 'value': v} for k, v in EIGHT_GAUSSIAN_EXPERIMENT.items()
] + [
    {'parameter': 'sald_va_particles', 'value': eight_profile.n_particles},
    {'parameter': 'doit_particles_equal_budget', 'value': eight_doit_profile.n_particles},
])


In [ ]:
eight_setup_fig = make_eight_gaussian_setup_figure(eight_cfg, eight_grid)
eight_setup_fig.savefig(
    OUTPUT_DIR / f'eight_gaussian_target_setup_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.png',
    dpi=240,
    bbox_inches='tight',
)
eight_setup_fig


In [ ]:
eight_sald_suite = run_eight_gaussian_algorithm(
    'SALD',
    EIGHT_GAUSSIAN_EXPERIMENT['r_values'],
    eight_cfg,
    eight_profile,
    eight_grid,
    seed=EIGHT_GAUSSIAN_EXPERIMENT['seed'],
    verbose=True,
)
eight_va_suite = run_eight_gaussian_algorithm(
    'VA-SALD',
    EIGHT_GAUSSIAN_EXPERIMENT['r_values'],
    eight_cfg,
    eight_profile,
    eight_grid,
    seed=EIGHT_GAUSSIAN_EXPERIMENT['seed'],
    verbose=True,
)
eight_doit_suite = run_eight_gaussian_algorithm(
    'DOIT',
    EIGHT_GAUSSIAN_EXPERIMENT['r_values'],
    eight_cfg,
    eight_doit_profile,
    eight_grid,
    seed=EIGHT_GAUSSIAN_EXPERIMENT['seed'],
    doit_cfg=eight_doit_cfg,
    verbose=True,
)

eight_sald_suite['summary_df'].to_csv(OUTPUT_DIR / f'eight_gaussian_sald_summary_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.csv', index=False)
eight_va_suite['summary_df'].to_csv(OUTPUT_DIR / f'eight_gaussian_va_summary_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.csv', index=False)
eight_doit_suite['summary_df'].to_csv(OUTPUT_DIR / f'eight_gaussian_doit_summary_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.csv', index=False)
eight_sald_suite['trajectory_df'].to_csv(OUTPUT_DIR / f'eight_gaussian_sald_trajectory_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.csv', index=False)
eight_va_suite['trajectory_df'].to_csv(OUTPUT_DIR / f'eight_gaussian_va_trajectory_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.csv', index=False)
eight_doit_suite['trajectory_df'].to_csv(OUTPUT_DIR / f'eight_gaussian_doit_trajectory_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.csv', index=False)

pd.concat([
    eight_sald_suite['summary_df'].assign(method='SALD'),
    eight_va_suite['summary_df'].assign(method='VA-SALD'),
    eight_doit_suite['summary_df'].assign(method='DOIT'),
], ignore_index=True)[['method', 'r', 'kl_to_target', 'mean_guidance', 'mean_penalty', 'n_particles', 'n_steps', 'wall_clock_sec']]


In [ ]:
eight_sald_overview_fig = make_eight_gaussian_algorithm_overview_figure(
    eight_sald_suite,
    algorithm_name='SALD',
    selected_r=EIGHT_GAUSSIAN_EXPERIMENT['selected_r_for_plots'],
)
eight_sald_overview_fig.savefig(
    OUTPUT_DIR / f'eight_gaussian_sald_overview_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.png',
    dpi=240,
    bbox_inches='tight',
)

eight_va_overview_fig = make_eight_gaussian_algorithm_overview_figure(
    eight_va_suite,
    algorithm_name='VA-SALD',
    selected_r=EIGHT_GAUSSIAN_EXPERIMENT['selected_r_for_plots'],
)
eight_va_overview_fig.savefig(
    OUTPUT_DIR / f'eight_gaussian_va_overview_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.png',
    dpi=240,
    bbox_inches='tight',
)

eight_doit_overview_fig = make_eight_gaussian_algorithm_overview_figure(
    eight_doit_suite,
    algorithm_name='DOIT',
    selected_r=EIGHT_GAUSSIAN_EXPERIMENT['selected_r_for_plots'],
)
eight_doit_overview_fig.savefig(
    OUTPUT_DIR / f'eight_gaussian_doit_overview_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.png',
    dpi=240,
    bbox_inches='tight',
)
eight_doit_overview_fig

In [ ]:
eight_compare_fig = make_eight_gaussian_comparison_figure(
    eight_sald_suite,
    eight_va_suite,
    eight_doit_suite,
)
eight_compare_fig.savefig(
    OUTPUT_DIR / f'three_algorithm_comparison_eight_gaussian_{EIGHT_GAUSSIAN_EXPERIMENT["run_name"]}.png',
    dpi=240,
    bbox_inches='tight',
)
eight_compare_fig


### What To Look For in the Guided Experiment

The most important diagnostics are now:

- the final KL to the guided target $q_T^X$,
- the terminal-target KL $KL(\rho_s \| q_T^X)$ tracked along the full trajectory,
- the mean guidance objective $\mathbb{E}[-f(X_s)]$, which should increase, plus `mean_penalty` for the raw decreasing penalty,
- the final sample panels versus the corresponding guided terminal target.

The guidance scale is fixed at $\lambda=1$ in all sections below. To change it, edit `lambda_guidance` in `GUIDED_EXPERIMENT`; the VA-SALD and DOIT sections copy the same setting for fair comparison.
